# Problem 2: Defining the Random Seed
I will add a seed to allow verification of the results

In [3]:
# Set seeds for reproducibility
import torch
import numpy as np

seed = 7
torch.manual_seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # This forces deterministic behavior for CuDNN
    torch.backends.cudnn.deterministic = True 
    torch.backends.cudnn.benchmark = False

# Problem 2: Tiny Shakespeare Dataset & Hyperparameter Tuning
This cell scales up our approach using the ~1.1 million character Tiny Shakespeare dataset. It tests baseline LSTM and GRU models at sequence lengths 20 and 30. It also explores the impact of hyperparameters by testing deeper networks (2 layers), wider networks (256 hidden states), intermediate fully connected layers, and extended sequence lengths (50).

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import requests
import time

# Check for CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# ---------------------------------------------------------
# 1. Dynamic Data Loader Wrapper
# ---------------------------------------------------------
# Download dataset once globally to save bandwidth
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text  

chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}
encoded_text = [char_to_int[ch] for ch in text]

class CharDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        return self.sequences[index], self.targets[index]

def get_data_loaders(sequence_length, batch_size=256):
    """Wraps the professor's data prep so we can change seq_length dynamically."""
    sequences = []
    targets = []
    
    # Create sequences
    for i in range(0, len(encoded_text) - sequence_length):
        seq = encoded_text[i:i+sequence_length]
        target = encoded_text[i+sequence_length]
        sequences.append(seq)
        targets.append(target)

    # Convert to tensors
    sequences_tensor = torch.tensor(sequences, dtype=torch.long)
    targets_tensor = torch.tensor(targets, dtype=torch.long)

    dataset = CharDataset(sequences_tensor, targets_tensor)

    train_size = int(len(dataset) * 0.8)
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size, num_workers=0)
    test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size, num_workers=0)
    
    return train_loader, test_loader

# ---------------------------------------------------------
# 2. Configurable Model (Handles layers, hidden states, FC net)
# ---------------------------------------------------------
class ShakespeareModel(nn.Module):
    def __init__(self, model_type, vocab_size, hidden_size, num_layers, fc_hidden_size=None):
        super(ShakespeareModel, self).__init__()
        self.model_type = model_type
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        
        if self.model_type == 'LSTM':
            self.rnn = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        elif self.model_type == 'GRU':
            self.rnn = nn.GRU(hidden_size, hidden_size, num_layers, batch_first=True)
            
        # Optional hidden fully connected layer to test hyperparameter adjustments
        self.use_fc_hidden = fc_hidden_size is not None
        if self.use_fc_hidden:
            self.fc1 = nn.Linear(hidden_size, fc_hidden_size)
            self.relu = nn.ReLU()
            self.fc2 = nn.Linear(fc_hidden_size, vocab_size)
        else:
            self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded)
        output = output[:, -1, :] 
        
        if self.use_fc_hidden:
            output = self.relu(self.fc1(output))
            output = self.fc2(output)
        else:
            output = self.fc(output)
            
        return output

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ---------------------------------------------------------
# 3. Inference Function (Tracks Time & Output)
# ---------------------------------------------------------
def generate_text(model, seed_text, generate_length, seq_length):
    model.eval()
    generated = seed_text
    
    if len(seed_text) < seq_length:
        current_seq = seed_text.rjust(seq_length, ' ')
    else:
        current_seq = seed_text[-seq_length:]
        
    start_time = time.time()
    with torch.no_grad():
        for _ in range(generate_length):
            input_seq = torch.tensor([char_to_int.get(c, 0) for c in current_seq], dtype=torch.long).unsqueeze(0).to(device)
            prediction = model(input_seq)
            predicted_index = torch.argmax(prediction, dim=1).item()
            predicted_char = int_to_char[predicted_index]
            
            generated += predicted_char
            current_seq = current_seq[1:] + predicted_char
            
    inference_time = time.time() - start_time
    return generated, inference_time

# ---------------------------------------------------------
# 4. Experiment Configurations & Main Loop
# ---------------------------------------------------------
# We will define dictionaries of experiments to fulfill all the prompt's requests
experiments = [
    # Base tests for Seq 20 and 30
    {"name": "LSTM_Seq20_Base", "type": "LSTM", "seq_len": 20, "hidden": 128, "layers": 1, "fc_hidden": None},
    {"name": "GRU_Seq20_Base",  "type": "GRU",  "seq_len": 20, "hidden": 128, "layers": 1, "fc_hidden": None},
    {"name": "LSTM_Seq30_Base", "type": "LSTM", "seq_len": 30, "hidden": 128, "layers": 1, "fc_hidden": None},
    {"name": "GRU_Seq30_Base",  "type": "GRU",  "seq_len": 30, "hidden": 128, "layers": 1, "fc_hidden": None},
    
    # Hyperparameter Sweeps (Adjusting layers, hidden states, and FC network)
    {"name": "LSTM_Seq30_Deep", "type": "LSTM", "seq_len": 30, "hidden": 128, "layers": 2, "fc_hidden": None}, # 2 Layers
    {"name": "GRU_Seq30_Wide",  "type": "GRU",  "seq_len": 30, "hidden": 256, "layers": 1, "fc_hidden": None}, # Wider hidden state
    {"name": "GRU_Seq30_ExtraFC", "type": "GRU", "seq_len": 30, "hidden": 128, "layers": 1, "fc_hidden": 64},  # Added FC Layer
    
    # What if we increase sequence length to 50?
    {"name": "LSTM_Seq50_Base", "type": "LSTM", "seq_len": 50, "hidden": 128, "layers": 1, "fc_hidden": None},
]

epochs = 5  # Keeping it low due to dataset size. Increase if you have time.
learning_rate = 0.005
results = {}

print("Starting experiments. This will take a while depending on your GPU...")

for exp in experiments:
    print(f"\n{'='*50}\nRunning Experiment: {exp['name']}\n{'='*50}")
    
    # Get DataLoaders for specific sequence length
    train_loader, test_loader = get_data_loaders(exp['seq_len'])
    
    # Initialize Model
    model = ShakespeareModel(
        model_type=exp['type'], 
        vocab_size=vocab_size, 
        hidden_size=exp['hidden'], 
        num_layers=exp['layers'], 
        fc_hidden_size=exp['fc_hidden']
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    num_params = count_parameters(model)
    print(f"Model Complexity: {num_params} parameters")
    
    start_time = time.time()
    
    # Training Loop
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch_idx, (inputs, targets_batch) in enumerate(train_loader):
            inputs, targets_batch = inputs.to(device), targets_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets_batch)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            # Print occasionally to show it hasn't frozen
            if (batch_idx + 1) % 1000 == 0:
                print(f"  Epoch [{epoch+1}/{epochs}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
                
        avg_train_loss = total_loss / len(train_loader)
        
        # Validation Loop (Run at the end of each epoch)
        model.eval()
        correct = 0
        total = 0
        val_loss = 0
        with torch.no_grad():
            for inputs, targets_batch in test_loader:
                inputs, targets_batch = inputs.to(device), targets_batch.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets_batch)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += targets_batch.size(0)
                correct += (predicted == targets_batch).sum().item()
                
        val_accuracy = correct / total
        avg_val_loss = val_loss / len(test_loader)
        
        print(f"-> Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")

    train_time = time.time() - start_time
    
    # Generate Text & Measure Inference Time
    seed = "ROMEO:"
    gen_text, inf_time = generate_text(model, seed, generate_length=100, seq_length=exp['seq_len'])
    
    print(f"\nGenerated Text:\n{gen_text}\n")
    
    # Store results
    results[exp['name']] = {
        'Train Time (s)': train_time,
        'Inference Time (s)': inf_time,
        'Params': num_params,
        'Train Loss': avg_train_loss,
        'Val Accuracy': val_accuracy,
        'Sample Text': gen_text.replace('\n', ' ')[:50] + "..."
    }

# ---------------------------------------------------------
# 5. Final Report Generator
# ---------------------------------------------------------
print("\n" + "="*110)
print(f"{'Experiment Name':<18} | {'Params':<8} | {'Train Time':<12} | {'Inf Time':<10} | {'Train Loss':<12} | {'Val Acc':<8}")
print("="*110)
for name, data in results.items():
    print(f"{name:<18} | {data['Params']:<8} | {data['Train Time (s)']:<12.2f} | {data['Inference Time (s)']:<10.4f} | {data['Train Loss']:<12.4f} | {data['Val Accuracy']:<8.4f}")
print("="*110)

Using device: cuda

Starting experiments. This will take a while depending on your GPU...

Running Experiment: LSTM_Seq20_Base
Model Complexity: 148801 parameters
  Epoch [1/5], Batch [1000/3486], Loss: 1.7826
  Epoch [1/5], Batch [2000/3486], Loss: 1.6525
  Epoch [1/5], Batch [3000/3486], Loss: 1.7787
-> Epoch 1 Summary | Train Loss: 1.7496 | Val Loss: 1.6126 | Val Acc: 0.5165
  Epoch [2/5], Batch [1000/3486], Loss: 1.5710
  Epoch [2/5], Batch [2000/3486], Loss: 1.6053
  Epoch [2/5], Batch [3000/3486], Loss: 1.5366
-> Epoch 2 Summary | Train Loss: 1.5671 | Val Loss: 1.5559 | Val Acc: 0.5290
  Epoch [3/5], Batch [1000/3486], Loss: 1.4291
  Epoch [3/5], Batch [2000/3486], Loss: 1.5104
  Epoch [3/5], Batch [3000/3486], Loss: 1.4860
-> Epoch 3 Summary | Train Loss: 1.5225 | Val Loss: 1.5275 | Val Acc: 0.5374
  Epoch [4/5], Batch [1000/3486], Loss: 1.5268
  Epoch [4/5], Batch [2000/3486], Loss: 1.4300
  Epoch [4/5], Batch [3000/3486], Loss: 1.4874
-> Epoch 4 Summary | Train Loss: 1.5038 | 

## Problem 2: The "Champion" Model 
Based on the previous hyperparameter sweep, this cell trains the most promising architecture (LSTM, Sequence Length 30, 2 Hidden Layers) for an extended 30 epochs. We introduce a dropout rate of 20% to prevent overfitting and a slightly lower learning rate to encourage stable convergence.

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import requests
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# 1. Prepare Data (Downloading once just in case this is a fresh kernel)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text  
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}
encoded_text = [char_to_int[ch] for ch in text]

# 2. Dataset & Loader
class CharDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets
    def __len__(self): return len(self.sequences)
    def __getitem__(self, index): return self.sequences[index], self.targets[index]

seq_length = 30
batch_size = 256
sequences = []
targets = []

for i in range(0, len(encoded_text) - seq_length):
    sequences.append(encoded_text[i:i+seq_length])
    targets.append(encoded_text[i+seq_length])

dataset = CharDataset(torch.tensor(sequences, dtype=torch.long), torch.tensor(targets, dtype=torch.long))
train_size = int(len(dataset) * 0.8)
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, len(dataset) - train_size])

train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)

# 3. The "Champion" Model with Dropout
class ChampionLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers, dropout_rate):
        super(ChampionLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        # Added dropout here to prevent overfitting during longer training
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True, dropout=dropout_rate)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        output = self.fc(output[:, -1, :]) 
        return output

hidden_size = 128
num_layers = 2
dropout_rate = 0.2
epochs = 30
learning_rate = 0.002 # Slightly lower for a longer, more stable training run

model = ChampionLSTM(vocab_size, hidden_size, num_layers, dropout_rate).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"{'='*50}\nTRAINING CHAMPION MODEL (30 Epochs)\n{'='*50}")
start_time = time.time()

# 4. Training Loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for inputs, targets_batch in train_loader:
        inputs, targets_batch = inputs.to(device), targets_batch.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
            
    avg_train_loss = total_loss / len(train_loader)
    
    # Validation
    model.eval()
    correct, total, val_loss = 0, 0, 0
    with torch.no_grad():
        for inputs, targets_batch in test_loader:
            inputs, targets_batch = inputs.to(device), targets_batch.to(device)
            outputs = model(inputs)
            val_loss += criterion(outputs, targets_batch).item()
            _, predicted = torch.max(outputs.data, 1)
            total += targets_batch.size(0)
            correct += (predicted == targets_batch).sum().item()
            
    val_accuracy = correct / total
    avg_val_loss = val_loss / len(test_loader)
    
    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")

print(f"\nTotal Training Time: {(time.time() - start_time) / 60:.2f} minutes")

# 5. Text Generation
print(f"\n{'='*50}\nCHAMPION TEXT GENERATION\n{'='*50}")
seed_text = "ROMEO:\nWhat light through yonder"
generate_length = 250
generated = seed_text
current_seq = seed_text[-seq_length:] if len(seed_text) >= seq_length else seed_text.rjust(seq_length, ' ')

model.eval()
with torch.no_grad():
    for _ in range(generate_length):
        input_seq = torch.tensor([char_to_int.get(c, 0) for c in current_seq], dtype=torch.long).unsqueeze(0).to(device)
        prediction = model(input_seq)
        predicted_char = int_to_char[torch.argmax(prediction, dim=1).item()]
        generated += predicted_char
        current_seq = current_seq[1:] + predicted_char
        
print(generated)
print(f"{'='*50}")

Using device: cuda

TRAINING CHAMPION MODEL (30 Epochs)
Epoch 01/30 | Train Loss: 1.8409 | Val Loss: 1.5882 | Val Acc: 0.5232
Epoch 02/30 | Train Loss: 1.5710 | Val Loss: 1.5020 | Val Acc: 0.5457
Epoch 03/30 | Train Loss: 1.5096 | Val Loss: 1.4641 | Val Acc: 0.5543
Epoch 04/30 | Train Loss: 1.4771 | Val Loss: 1.4401 | Val Acc: 0.5610
Epoch 05/30 | Train Loss: 1.4555 | Val Loss: 1.4215 | Val Acc: 0.5666
Epoch 06/30 | Train Loss: 1.4412 | Val Loss: 1.4127 | Val Acc: 0.5684
Epoch 07/30 | Train Loss: 1.4303 | Val Loss: 1.4042 | Val Acc: 0.5716
Epoch 08/30 | Train Loss: 1.4211 | Val Loss: 1.3958 | Val Acc: 0.5727
Epoch 09/30 | Train Loss: 1.4133 | Val Loss: 1.3912 | Val Acc: 0.5727
Epoch 10/30 | Train Loss: 1.4059 | Val Loss: 1.3884 | Val Acc: 0.5742
Epoch 11/30 | Train Loss: 1.4020 | Val Loss: 1.3855 | Val Acc: 0.5758
Epoch 12/30 | Train Loss: 1.3979 | Val Loss: 1.3786 | Val Acc: 0.5777
Epoch 13/30 | Train Loss: 1.3942 | Val Loss: 1.3757 | Val Acc: 0.5776
Epoch 14/30 | Train Loss: 1.3906 |